# 03 - Feature Engineering

Turning the cleaned weekly counts into something a model can learn from.

The setup: **standing at week t, predict week t+1** for each district. The features are this week's values, and the targets come from next week.

Two targets, so either kind of model can be trained from the same file:
- `target_log_next` - next week's cases on the log scale (regression)
- `risk_next` - next week's risk level: low / medium / high (classification)

The two features, `log_cases` and `district_code`, were already made in notebook 02. This notebook adds the targets and splits the data into train and test.

**Input:** `dataset/processed/02_cleaned_weekly_data.csv`
**Output:** `dataset/processed/train.csv`, `dataset/processed/test.csv` and `dataset/processed/03_risk_thresholds.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../dataset")
IN_PATH = DATA_DIR / "processed" / "02_cleaned_weekly_data.csv"
TRAIN_PATH = DATA_DIR / "processed" / "train.csv"
TEST_PATH = DATA_DIR / "processed" / "test.csv"
THRESH_PATH = DATA_DIR / "processed" / "03_risk_thresholds.csv"

TRAIN_END = "2023-12-31"  # train on 2007-2023, test on 2024-2026

In [2]:
df = pd.read_csv(IN_PATH, parse_dates=["start_date", "end_date"])
df = df.sort_values(["district", "start_date"]).reset_index(drop=True)

# only used inside this notebook (risk thresholds + the final split), not saved
df["split"] = np.where(df["start_date"] <= TRAIN_END, "train", "test")

print(df.shape)
df["split"].value_counts()

(26728, 11)


split
train    23114
test      3614
Name: count, dtype: int64

## 1. Targets

**Regression:** `target_log_next` = next week's `log_cases` (turn predictions back into cases with `np.expm1`).

**Classification:** a big district like Colombo has hundreds of cases every week, while Mullaitivu usually has 0-2, so one national cut-off doesn't work. Instead each district gets its own thresholds from its **training years only** (so the test period doesn't leak in):
- **low:** up to the district's 75th percentile
- **medium:** above the 75th, up to the 90th percentile
- **high:** above the 90th percentile

I also keep `target_imputed`, which says if next week's value was estimated in cleaning, so those weeks can be left out when scoring the model.

In [3]:
by_district = df.groupby("district")

df["cases_next"] = by_district["cases"].shift(-1)
df["target_log_next"] = by_district["log_cases"].shift(-1)
df["target_imputed"] = by_district["is_imputed"].shift(-1)

thresholds = (
    df[df["split"] == "train"]
    .groupby("district", observed=True)["cases"]
    .quantile([0.75, 0.90])
    .unstack()
    .set_axis(["medium_above", "high_above"], axis=1)
)
thresholds.sort_values("high_above")

,medium_above,high_above
district,,
Mullaitivu,2.0,5.0
Kilinochchi,2.0,6.0
Ampara,4.0,9.0
Mannar,3.0,10.0
NuwaraEliya,6.0,11.0
Polonnaruwa,8.0,14.0
Vavuniya,5.0,15.0
Monaragala,9.0,20.0
Anuradhapura,13.0,26.0


In [4]:
t = df[["district"]].join(thresholds, on="district")

df["risk_next"] = np.select(
    [df["cases_next"] > t["high_above"], df["cases_next"] > t["medium_above"]],
    ["high", "medium"],
    default="low",
)
df["risk_next"] = df["risk_next"].where(df["cases_next"].notna())

pd.crosstab(df["split"], df["risk_next"], normalize="index").round(2)

risk_next,high,low,medium
split,,,
test,0.16,0.54,0.30
train,0.10,0.76,0.14


Training comes out at about 76% low / 14% medium / 10% high, as expected from the percentiles. The test years (2024-2026) have far more medium and high weeks (30% / 16%) because case levels were higher recently. That's worth remembering when looking at test scores.

## 2. Drop incomplete rows

The last week of each district has no next week to predict, so it's dropped.

In [5]:
FEATURES = ["log_cases", "district_code"]
TARGETS = ["target_log_next", "risk_next"]

before = len(df)
df = df.dropna(subset=FEATURES + TARGETS).reset_index(drop=True)
print("dropped", before - len(df), "rows ->", df.shape)

dropped 26 rows -> (26702, 15)


## Checks

In [6]:
assert df[FEATURES + TARGETS].notna().all().all()
assert not df.duplicated(["district", "start_date"]).any()

# the target really is the next week's value
nxt = df.groupby("district", observed=True)["log_cases"].shift(-1)
gap = df.groupby("district", observed=True)["start_date"].shift(-1) - df["start_date"]
one_week = gap == pd.Timedelta(days=7)
assert (df.loc[one_week, "target_log_next"] == nxt[one_week]).all()

print("all checks passed")

all checks passed


## 3. Train / test files

Last step: split by date into two files. Everything up to the end of 2023 is training, 2024 onwards is test. No shuffling, since it's a time series and the model must never see the future during training.

In [7]:
cols = ["start_date", "year", "week", "district", "cases", "is_imputed", "dup_week",
        *FEATURES, "cases_next", "target_log_next", "risk_next", "target_imputed"]

df = df.sort_values(["start_date", "district"])
train = df[df["split"] == "train"][cols]
test = df[df["split"] == "test"][cols]

assert len(train) + len(test) == len(df)
assert train["start_date"].max() < test["start_date"].min()

for name, part in [("train", train), ("test", test)]:
    print(f"{name}: {len(part)} rows, {part['start_date'].min():%Y-%m-%d} to {part['start_date'].max():%Y-%m-%d}")

train: 23114 rows, 2006-12-23 to 2023-12-30
test: 3588 rows, 2024-01-06 to 2026-08-22


In [8]:
train.to_csv(TRAIN_PATH, index=False, date_format="%Y-%m-%d")
test.to_csv(TEST_PATH, index=False, date_format="%Y-%m-%d")
thresholds.to_csv(THRESH_PATH)
print("saved", TRAIN_PATH.name, TEST_PATH.name, THRESH_PATH.name)

saved train.csv test.csv 03_risk_thresholds.csv


## Summary

| Feature | What it is |
|---|---|
| `log_cases` | log1p of this week's cases (from notebook 02) |
| `district_code` | which district (from notebook 02) |

**Targets:** `target_log_next` (regression) and `risk_next` (low/medium/high by each district's own 75th/90th percentile from 2007-2023).

**Files:** `train.csv` (2007-2023) and `test.csv` (2024-2026), ready for modelling. The thresholds are saved separately in `03_risk_thresholds.csv` so the dashboard can turn predicted case numbers into the same risk levels.